# Linear model

Score: 0.15503 With a prior EDA we preprocess our data and train our model.
Linear models are easy to interpret, but they require more data preprocessing and rely on many hypothesis

In [284]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import plotly.express as px
import seaborn as sns

# Load dataset
df = pd.read_csv('train.csv') 


## 1.1 convert types

Its better to convert categorical variables to "category" to save memory, but since there are a few we convert to object for generic and simple handeling 

In [285]:
df[['Id', 'MSSubClass', 'MoSold', 'YearBuilt', 'YearRemodAdd', 'YrSold']] = df[['Id', 'MSSubClass', 'MoSold', 'YearBuilt', 'YearRemodAdd', 'YrSold']].astype('object')

## 1.2 drop variables with most values missing

Linear regression doesn't handle well rare categories

In [286]:
threshold = 0.30  # 30%
df = df.loc[:, df.isna().mean() <= threshold]

threshold = 0.30  # 30%
missing_ratio = df.isna().mean()

# Columns to drop (more than 30% missing)
dropped_columns_most_missing = missing_ratio[missing_ratio > threshold].index.tolist()

# Drop from training data
df = df.loc[:, missing_ratio <= threshold]

In [287]:
df.select_dtypes(include=[ 'object']).columns

Index(['Id', 'MSSubClass', 'MSZoning', 'Street', 'LotShape', 'LandContour',
       'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1',
       'Condition2', 'BldgType', 'HouseStyle', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual',
       'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
       'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir',
       'Electrical', 'KitchenQual', 'Functional', 'GarageType', 'GarageFinish',
       'GarageQual', 'GarageCond', 'PavedDrive', 'MoSold', 'YrSold',
       'SaleType', 'SaleCondition'],
      dtype='object')

## 1.3 Fill missing values with "Missing" for categorical variables

In [288]:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# Fill missing values with "Missing"
df[cat_cols] = df[cat_cols].fillna("Missing")



C:\Users\berra\AppData\Local\Temp\ipykernel_10228\1901938063.py:5: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



## 1.4 Fill missing values with the mean for quantitative variables

In [289]:
#save numeric means for missing test values
means = df.select_dtypes(include='number').mean()

#imputer numeric par moyenne dans train
df = df.fillna(df.select_dtypes(include='number').mean())

## 1.5 create variables for clarity HouseAge, YearsSinceRemod...

In [290]:
df_dropped = df.copy()

#'YearBuilt', 'YearRemodAdd, 'MoSold', 'YrSold' convert to quantitative
# List of columns to convert
columns_to_convert = ['YearBuilt', 'YearRemodAdd', 'MoSold', 'YrSold']

# Convert each to integer safely
for col in columns_to_convert:
    df_dropped.loc[:, col] = df_dropped[col].astype(int)

# Set reference year
reference_year = df_dropped['YrSold'].max()

# Create quantitative features safely
df_dropped.loc[:, 'HouseAge'] = reference_year - df_dropped['YearBuilt']
df_dropped.loc[:, 'YearsSinceRemod'] = reference_year -df_dropped['YearRemodAdd']
df_dropped.loc[:, 'TimeIndex'] = (
    (df_dropped['YrSold'] - df_dropped['YrSold'].min()) * 12 + df_dropped['MoSold']
)

df_dropped = df_dropped.drop([
    'YearBuilt', 'YearRemodAdd', 'YrSold', 'MoSold'
], axis=1) 

# Convert object columns to numeric, forcing errors to NaN
cols_to_convert = ['HouseAge', 'YearsSinceRemod', 'TimeIndex']
df_dropped[cols_to_convert] = df_dropped[cols_to_convert].apply(pd.to_numeric, errors='coerce')

## Transform categorical variables

MSSubclass seems to be an important variable, it correlates well with SalePrice (see box plots, medium, variance in eda notebook). For example: 60( 2-STORY 1946 & NEWER) and 120 (1-STORY PUD (Planned Unit Development) - 1946 & NEWER) seem to have very high prices (domaine knowledge explain that well). We can create less sparse variables using these elements: category size, correlation with price, domain knoledge. This will give us 3 principal variables.

In [291]:
percentages = df['MSSubClass'].value_counts(normalize=True) * 100
print(percentages)

ms = df["MSSubClass"]

df["story_group"] = np.where(ms.isin([20, 30, 40, 120]), "1", "more")
df["is_pud"]      = ms.isin([120, 150, 160, 180]).astype("int8")
df["age_band"]    = np.select(
    [ms.isin([20, 60, 120, 160]), ms.isin([30, 70])],
    ["new", "old"],
    default="all"
)

MSSubClass
20     36.712329
60     20.479452
50      9.863014
120     5.958904
30      4.726027
160     4.315068
70      4.109589
80      3.972603
90      3.561644
190     2.054795
85      1.369863
75      1.095890
45      0.821918
180     0.684932
40      0.273973
Name: proportion, dtype: float64


Using Price correlation alone can be harmful (data leakage), this is why this why this method is better. We can think about other methods like target encoding with OOF (Out-Of-Fold), but we keep things simple for interpretation. We can thing about this method for variables with a lot of categories.

The treatment of MSSubClass has made our model better by 0.025, other categorical variables can be treated like neighbourhood... (grouping close neighbourhoods...)

## 1.6 Correct skewness

For variables with a significant skewness from the test pandas.DataFrame.skew()

In [292]:
#case 1: only binary no log

#Most houses don't have low-quality finished square footage.
#Very few do, with highly varied amounts.
cols_to_binary_only_0 = ['BsmtHalfBath', 'EnclosedPorch', 'ScreenPorch']

for col in cols_to_binary_only_0:
    df_dropped[f'Has{col}'] = (df_dropped[col] > 0).astype(int)
    df_dropped.drop(columns=[col], inplace=True)

# = 1 or not

df_dropped['HasKitchen'] = (df_dropped['KitchenAbvGr'] == 1).astype(int)
df_dropped.drop(columns=['KitchenAbvGr'], inplace=True)


#case 2: binary+log

cols = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', '2ndFlrSF', 'WoodDeckSF', 'OpenPorchSF']  # Replace with actual column names

for col in cols:
    df_dropped[f'Has{col}'] = (df_dropped[col] > 0).astype(int)
    df_dropped[f'{col}_log'] = np.log1p(df_dropped[col])

#case 3: just drop 
df_dropped.drop(columns=['LowQualFinSF','3SsnPorch', 'PoolArea', 'MiscVal'], inplace=True)

# case 4: log only

df_dropped.drop(columns=[ 'TimeIndex'], inplace=True)# cyclique, pas significative

# List of variables to transform
vars_to_log = ['LotFrontage', 'LotArea', 'TotalBsmtSF', 'GrLivArea',  'SalePrice', 'BsmtUnfSF', '1stFlrSF']

# Create log-transformed versions with "_log" suffix
for col in vars_to_log:
    df_dropped[col + '_log'] = np.log1p(df_dropped[col])  # log1p handles zero safely

# Drop the original columns
df_dropped.drop(columns=vars_to_log, inplace=True)

## 1.7 Cap outliers

In [293]:
# cap outliers
#outliers cap

num_features = [col for col in df_dropped.select_dtypes(include='number') if col != 'Id']
 
# Store the limits for each column
caps = {}

for col in num_features:
    q_low = df_dropped[col].quantile(0.01)
    q_high = df_dropped[col].quantile(0.99)
    
    # Save the thresholds
    caps[col] = (q_low, q_high)
    
    # Apply clipping
    df_dropped[col] = df_dropped[col].clip(lower=q_low, upper=q_high)



## 1.8 Categorical variable (merge rare categories into the most frequent)

In [294]:
def clean_categorical_variables(
    df, 
    target_col='SalePrice_log', 
    threshold=0.03, 
    tol=0.10, 
    min_count=30
):
    """
    Cleans all categorical variables:
    - Fills missing values
    - Groups rare non-predictive categories into 'Other'
    - Merges 'Other' into closest price category if too small

    Returns modified DataFrame.
    """
    df = df.copy()
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    global_mean = df[target_col].mean()
    mappings = {}
    for col in cat_cols:

        # Step 2: Frequency and stats
        freq = df[col].value_counts(normalize=True)
        rare_cats = freq[freq < threshold].index
        stats = df.groupby(col)[target_col].agg(['count', 'mean'])

        # Step 3: Decide which rare categories to keep
        keep_rare = []
        group_rare = []

        for cat in rare_cats:
            count = stats.loc[cat, 'count']
            mean = stats.loc[cat, 'mean']
            deviation = abs(mean - global_mean) / global_mean

            if count >= min_count and deviation > tol:
                keep_rare.append(cat)
            else:
                group_rare.append(cat)

        # Step 4: Replace rare with 'Other'
        df[col] = df[col].apply(lambda x: 'Other' if x in group_rare else x)

        final_merge_target = None
        # Step 5: Merge 'Other' if too small

        if 'Other' in df[col].values:
            other_mask = df[col] == 'Other'
            if other_mask.sum() < min_count:
                # Merge 'Other' into the most frequent existing category (excluding 'Other')
                final_merge_target = df.loc[~other_mask, col].value_counts().idxmax()
                df.loc[other_mask, col] = final_merge_target
                print(f"'{col}': 'Other' merged into most frequent category '{final_merge_target}'")

 # Save mapping
        mappings[col] = {
            'group_rare': group_rare,
            'final_merge_target': final_merge_target
        }

    return df, mappings



df_rare_cat, cat_mappings = clean_categorical_variables(df_dropped)


#remove variables with one category

# Step 1: Identify columns dropped from training
cols_dropped_one_cat = df_dropped.columns[df_dropped.nunique(dropna=False) <= 1]
df_dropped = df_dropped.loc[:, df_dropped.nunique(dropna=False) > 1]

from sklearn.preprocessing import OneHotEncoder,LabelEncoder,RobustScaler

y_avant = df_dropped['SalePrice_log']

'MSZoning': 'Other' merged into most frequent category 'RL'
'Street': 'Other' merged into most frequent category 'Pave'
'Utilities': 'Other' merged into most frequent category 'AllPub'
'LotConfig': 'Other' merged into most frequent category 'Inside'
'LandSlope': 'Other' merged into most frequent category 'Gtl'
'Condition2': 'Other' merged into most frequent category 'Norm'
'RoofMatl': 'Other' merged into most frequent category 'CompShg'
'ExterQual': 'Other' merged into most frequent category 'TA'
'HeatingQC': 'Other' merged into most frequent category 'Ex'
'GarageQual': 'Other' merged into most frequent category 'TA'


In [295]:
df_dropped[[ 'MSSubClass']] = df_dropped[[ 'MSSubClass']].astype('object')

### Scaling

In [296]:
#df_dropped[df_dropped.select_dtypes(include='number').columns.drop('Id')] = RobustScaler().fit_transform(df_dropped[df_dropped.select_dtypes(include='number').columns.drop('Id')])
#df_dropped[df_dropped.select_dtypes(include='float').columns] = RobustScaler().fit_transform(df_dropped[df_dropped.select_dtypes(include='float').columns])

# Select float and int columns, excluding 'Id'
cols_to_scale = df_dropped.select_dtypes(include=['float', 'int']).columns.drop(['Id', 'SalePrice_log'])
scaler = RobustScaler()
# Apply RobustScaler
df_dropped[cols_to_scale] = scaler.fit_transform(df_dropped[cols_to_scale])

In [298]:
# As a plain Python list
df_dropped.select_dtypes(include=['int']).columns.tolist()

['Id']

In [299]:
# One-hot encode all object or category dtype columns
df_dropped = pd.get_dummies(df_dropped, drop_first=True)

## 1.9 Chose/train model

Test ridge et lasso models with différents paramètres to chose the best with cross validation to avoid over fitting

In [300]:
# from sklearn.model_selection import cross_val_score, KFold
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error
# from sklearn.ensemble import GradientBoostingRegressor

# # Step 1: Separate features and target
# X_train = df_dropped.drop(columns=['Id', 'SalePrice_log'])  # Replace 'Price' with your actual target if named differently
# y = df_dropped['SalePrice_log']


### tester Ridge

In [301]:
# from sklearn.linear_model import RidgeCV

# alphas = np.logspace(-4, 4, 50)
# ridge_cv = RidgeCV(alphas=alphas, store_cv_values=True)
# ridge_cv.fit(X_train, y)

# print("Best alpha:", ridge_cv.alpha_)

# #the best
# from sklearn.linear_model import Ridge
# model = Ridge(alpha=7.9060432109076855)

# model.fit(X_train, y)

# # Step 4: Cross-validation setup
# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# # Step 5: Run cross-validation and evaluate using negative RMSE
# scores = cross_val_score(model, X_train, y, scoring='neg_root_mean_squared_error', cv=cv)

# # Step 6: Print results
# print("Cross-validated RMSE scores:", -scores)
# print("Average RMSE:", -scores.mean())

### Tester Lasso

In [302]:
# from sklearn.linear_model import LassoCV
# import numpy as np

# alphas = np.logspace(-4, 4, 50)
# lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=0)
# lasso_cv.fit(X_train, y)

# print("Best alpha:", lasso_cv.alpha_)

# #final
# from sklearn.linear_model import Lasso
# model = Lasso(alpha=0.0006551285568595509)

# model.fit(X_train, y)

# # Step 4: Cross-validation setup
# cv = KFold(n_splits=5, shuffle=True, random_state=42)

# # Step 5: Run cross-validation and evaluate using negative RMSE
# scores = cross_val_score(model, X_train, y, scoring='neg_root_mean_squared_error', cv=cv)

# # Step 6: Print results
# print("Cross-validated RMSE scores:", -scores)
# print("Average RMSE:", -scores.mean())

### Model-based feature selector

In [303]:
### 1.9.1 feature selection

from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


# Step 1: Separate features and target
X_train = df_dropped.drop(columns=['Id', 'SalePrice_log'])  # Replace 'Price' with your actual target if named differently

In [304]:
y = y_avant

# 2. Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_train, y, test_size=0.2, random_state=42)

We use Model-based feature selector: it fits an estimator that has coef_ or feature_importances_, keeps features whose importance passes a threshold (mean), we use it because we have many sparse, correlated numeric features

In [305]:
# 3. Feature selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42).fit(X_train, y_train)

# 4. Select features with threshold="mean"
selector = SelectFromModel(lasso_cv, threshold="mean", prefit=True)

X_train_sel = selector.transform(X_train)

C:\Users\berra\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning:

X has feature names, but SelectFromModel was fitted without feature names



### Best model

In [306]:
# 5. RidgeCV model with cross-validation
alphas = np.logspace(-4, 4, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_sel, y_train)

# 6. Evaluate on test set
y_pred = ridge_cv.predict(X_train_sel)
r2 = r2_score(y_train, y_pred)
mse = mean_squared_error(y_train, y_pred)

print(f"Best alpha (RidgeCV): {ridge_cv.alpha_}")
print(f"Test R² score: {r2:.4f}")
print(f"Test MSE: {mse:.2f}")

Best alpha (RidgeCV): 3.727593720314938
Test R² score: 0.9269
Test MSE: 0.01


### Selected features

In [307]:
# 7. View selected features
selected_features = X_train.columns[selector.get_support()]
print("Selected features:", selected_features.tolist())

Selected features: ['OverallQual', 'OverallCond', 'BsmtFinSF1', 'BsmtFullBath', 'FullBath', 'HalfBath', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'HouseAge', 'YearsSinceRemod', 'HasScreenPorch', 'HasKitchen', 'Has2ndFlrSF', 'LotArea_log', 'TotalBsmtSF_log', 'GrLivArea_log', '1stFlrSF_log', 'MSSubClass_30', 'MSSubClass_160', 'MSZoning_RM', 'LotShape_IR2', 'LandContour_HLS', 'LotConfig_CulDSac', 'Neighborhood_BrkSide', 'Neighborhood_Crawfor', 'Neighborhood_Edwards', 'Neighborhood_Mitchel', 'Neighborhood_NWAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'Condition1_Norm', 'Condition2_Norm', 'Exterior1st_BrkFace', 'ExterQual_TA', 'ExterCond_TA', 'Foundation_PConc', 'BsmtQual_Gd', 'BsmtQual_TA', 'BsmtCond_TA', 'BsmtExposure_Gd', 'BsmtExposure_No', 'BsmtFinType1_Unf', 'HeatingQC_Gd', 'HeatingQC_TA', 'CentralAir_Y', 'KitchenQual_Gd', 'KitchenQual_TA', 'Functional_Typ', 'SaleCondition_Normal', 'SaleCondition_Partial']


# Test set

In [308]:
# test

d_test = pd.read_csv('test.csv') 

In [309]:
d_test.loc[6, "LotFrontage"]

nan

## 2.1 convert types

In [310]:
d_test[['Id', 'MSSubClass', 'MoSold', 'YearBuilt', 'YearRemodAdd', 'YrSold']] = d_test[['Id', 'MSSubClass', 'MoSold', 'YearBuilt', 'YearRemodAdd', 'YrSold']].astype('object')

In [311]:
ms = d_test["MSSubClass"]

d_test["story_group"] = np.where(ms.isin([20, 30, 40, 120]), "1", "more")
d_test["is_pud"]      = ms.isin([120, 150, 160, 180]).astype("int8")
d_test["age_band"]    = np.select(
    [ms.isin([20, 60, 120, 160]), ms.isin([30, 70])],
    ["new", "old"],
    default="all"
)

## 2.2 drop variables with most values missing in train

In [314]:
d_test.drop(columns=dropped_columns_most_missing, inplace=True)

## 2.3 Fill missing values with "Missing" for categorical variables

In [316]:

# Identify categorical columns
cat_cols = d_test.select_dtypes(include=['object', 'category']).columns

# Fill missing values with "Missing"
d_test[cat_cols] = d_test[cat_cols].fillna("Missing")


C:\Users\berra\AppData\Local\Temp\ipykernel_10228\3436678766.py:5: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



## 2.4 Fill missing values with the mean for quantitative variables from the train

In [318]:
# Imputation dans le test set (numeric)
d_test.fillna(means, inplace=True)

## 2.5 create variables like the train

In [321]:
#'YearBuilt', 'YearRemodAdd, 'MoSold', 'YrSold' convert to quantitative
# List of columns to convert
columns_to_convert = ['YearBuilt', 'YearRemodAdd', 'MoSold', 'YrSold']

# Convert each to integer safely
for col in columns_to_convert:
    d_test.loc[:, col] = d_test[col].astype(int)

# Set reference year
reference_year = d_test['YrSold'].max()

# Create quantitative features safely
d_test.loc[:, 'HouseAge'] = reference_year - d_test['YearBuilt']
d_test.loc[:, 'YearsSinceRemod'] = reference_year - d_test['YearRemodAdd']


d_test = d_test.drop([
    'YearBuilt', 'YearRemodAdd', 'YrSold', 'MoSold'
], axis=1) 

# Convert object columns to numeric, forcing errors to NaN
cols_to_convert = ['HouseAge', 'YearsSinceRemod']
d_test[cols_to_convert] = d_test[cols_to_convert].apply(pd.to_numeric, errors='coerce')

## 2.6 Correct skewness

In [323]:
#case 1: only binary no log

#Most houses don't have low-quality finished square footage.
#Very few do, with highly varied amounts.
cols_to_binary_only_0 = ['BsmtHalfBath', 'EnclosedPorch', 'ScreenPorch']

for col in cols_to_binary_only_0:
    d_test[f'Has{col}'] = (d_test[col] > 0).astype(int)
    d_test.drop(columns=[col], inplace=True)

# = 1 or not

d_test['HasKitchen'] = (d_test['KitchenAbvGr'] == 1).astype(int)
d_test.drop(columns=['KitchenAbvGr'], inplace=True)


#case 2: binary+log

cols = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', '2ndFlrSF', 'WoodDeckSF', 'OpenPorchSF']  # Replace with actual column names

for col in cols:
    d_test[f'Has{col}'] = (d_test[col] > 0).astype(int)
    d_test[f'{col}_log'] = np.log1p(d_test[col])

#case 3: just drop 
d_test.drop(columns=['LowQualFinSF','3SsnPorch', 'PoolArea', 'MiscVal'], inplace=True)

# case 4: log only

# List of variables to transform
vars_to_log = ['LotFrontage', 'LotArea', 'TotalBsmtSF', 'GrLivArea',  'BsmtUnfSF', '1stFlrSF']


# Create log-transformed versions with "_log" suffix
for col in vars_to_log:
    d_test[col + '_log'] = np.log1p(d_test[col])  # log1p handles zero safely

# Drop the original columns
d_test.drop(columns=vars_to_log, inplace=True)

## 2.7 Cap outliers

In [325]:
num_features = [col for col in num_features if col != "SalePrice_log"]

for col in num_features:
    q_low, q_high = caps[col]
    d_test[col] = d_test[col].clip(lower=q_low, upper=q_high)


## 2.8 Categorical variable (merge rare categories into the most frequent in train)

In [328]:
def apply_cat_mapping_to_test(df_test, mappings):
    df_test = df_test.copy()

    for col, info in mappings.items():
        df_test[col] = df_test[col].fillna("Missing")

        # Replace rare categories with 'Other'
        df_test[col] = df_test[col].apply(lambda x: 'Other' if x in info['group_rare'] else x)

        # Merge 'Other' into target if needed
        if info['final_merge_target'] is not None:
            df_test[col] = df_test[col].replace('Other', info['final_merge_target'])

    return df_test
# Apply same mappings to test
d_test = apply_cat_mapping_to_test(d_test, cat_mappings)



# Step 2: Drop same columns from test set
d_test = d_test.drop(columns=cols_dropped_one_cat, errors='ignore')

### Scaling

In [331]:
d_test[[ 'MSSubClass']] = d_test[[ 'MSSubClass']].astype('object')
#d_test[d_test.select_dtypes(include='float').columns] = RobustScaler().fit_transform(d_test[d_test.select_dtypes(include='float').columns])
cols_to_scale = d_test.select_dtypes(include=['float', 'int']).columns.drop(['Id'])
d_test[cols_to_scale] = scaler.transform(d_test[cols_to_scale])

# 2.9 predict

In [333]:
# One-hot encode all object or category dtype columns
d_test = pd.get_dummies(d_test, drop_first=True)

In [335]:
X_test = d_test.drop(columns=['Id'], errors='ignore') 
# Align with training columns (very important!)
X_test_aligned = X_test.reindex(columns=X_train.columns, fill_value=0)

X_test_sel = selector.transform(X_test_aligned)

y_pred_log = ridge_cv.predict(X_test_sel)
y_pred = np.expm1(y_pred_log)  # reverse np.log1p()
d_test['SalePrice'] = y_pred
d_test[['Id', 'SalePrice']].to_csv("predictions_linear.csv", index=False)

C:\Users\berra\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning:

X has feature names, but SelectFromModel was fitted without feature names

